In [1]:
import os
from pathlib import Path
import meeplemate
WORKSPACE_PATH = Path(meeplemate.__file__).parent.parent
os.chdir(WORKSPACE_PATH)

In [2]:
from IPython.display import display, Markdown
import importlib

from numpy import full
import dev_system
from meeplemate.config import GameService
importlib.reload(dev_system)
from langchain_openai import ChatOpenAI
from dev_system import areload, get_service
from meeplemate.component_system import factory

await areload(
    ["qa_service", "game_service", "chat_model", "chunk_search_service_2", "full_page_store", "tokenizer"],
)
qa_service = get_service("qa_service")
game_service: GameService = get_service("game_service")
chat_model = get_service("chat_model")
chunk_search_service = get_service("chunk_search_service_2")
full_page_store = get_service("full_page_store")
tokenizer = get_service("tokenizer")

Loading configuration from: config-dev.yaml
Loading configuration from: config-dev.yaml
Reloading system...
System reloaded


In [3]:
game_id = "warhammer_5th_edition"
manifest = await game_service.get_manifest(game_id)

## Queston Analysis

What is the user asking? What are the main rule interactions? What sort of question is it (e.g. rule lookup, rule interaction)?

In [4]:
from re import sub
from uuid import uuid4

from langchain_core.runnables import RunnableConfig
from meeplemate.qa_graph import (
    build_analyze_question_graph,
    QuestionAnalysisContext,
    create_question_analysis_state
)
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.globals import set_debug

set_debug(False)

agent = build_analyze_question_graph(
    InMemorySaver(),
    chat_model,
    tokenizer,
)

context = QuestionAnalysisContext(
    manifest=manifest,
    chunk_search_service=chunk_search_service,
)
input = create_question_analysis_state(
    query="When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?"
    # query="What is the leadership value of a Grail Knight",
    # query="Do Grail Knights have any special abilities that prevent them from taking Break tests?"
    # query="How does Grail Virtue interacts with Break test?"
)

config: RunnableConfig = {"configurable": {"thread_id": str(uuid4())}}

output = await agent.ainvoke(input, context=context, config=config)

print("Classification:", output["classification"])

display(Markdown(output["analysis"]))
print("\n\n")
print("Subquestions:")
for interaction in output["subquestions"]:
    print(f"- {interaction}")

# print(output["reasoning"])
# # from pprint import pprint
# # pprint(output)
# # Display as Markdown
# display(Markdown(output["analysis"]["analysis"]))

# for subquestion in [subproblem["question"] for subproblem in output["analysis"]["subproblems"]]:
#     print(subquestion)

2026-02-10 20:01:24 [info     ] search_chunks called           search_terms=['Grail Knights combat rules', 'Green Dragon combat mechanics', 'Break test rules for Grail Knights', 'Grail Knights after combat loss', 'consequences of losing combat for Grail Knights'] token_budget=17291
2026-02-10 20:01:25 [info     ] Used tokens                    query=['Grail Knights combat rules', 'Green Dragon combat mechanics', 'Break test rules for Grail Knights', 'Grail Knights after combat loss', 'consequences of losing combat for Grail Knights'] tokens_used=10350 total_retrieved_count=21
2026-02-10 20:01:25 [info     ] Retrieved results              query='When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?' relevant_count=21 total_retrieved_count=21


/workspace/meeplemate/cassandra_util.py:39: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  return load(value) if value is not None else None


2026-02-10 20:01:30 [info     ] Subquestion identified         explanation='Understanding whether a combat loss inherently causes a break test is foundational. If no break test occurs upon losing combat, then the Grail Knights would not need one regardless of other factors. This depends on the core mechanics of combat resolution and morale in the Warhammer rulebook.' subquestion='Does losing a combat trigger a break test for any unit, regardless of type?'
2026-02-10 20:01:30 [info     ] Subquestion identified         explanation='Grail Knights possess the Grail Virtue, which explicitly states they are unaffected by psychology rules. This directly relates to whether they must take any psychological test—such as a Panic test—even after losing combat. Confirming this immunity is essential to answering whether they need a break test.' subquestion='Are Grail Knights immune to psychological effects such as panic or fear, and if so, under what conditions?'
2026-02-10 20:01:30 [info     ] Subq

The user's query asks whether Grail Knights must take a break test when they lose combat against a Green Dragon. This involves understanding multiple rules from the Bretonnia Army Book and Warhammer rulebooks, including:

- The definition of a 'break test' in the context of combat and morale.
- Whether losing a combat triggers a break test.
- The special rules for Green Dragons, particularly their ability to cause fear or panic.
- The immunity of Grail Knights to psychology (including panic) due to the Grail Virtue.
- The interaction between combat loss, psychological effects, and unit cohesion.

These rules are spread across different documents: the Bretonnia Army Book details Grail Knight virtues and combat behavior, while the Warhammer rulebook defines the mechanics of combat phases and psychology tests. The key interaction is whether a combat loss by a unit triggers a psychological test, and whether that test is negated by specific immunities like the Grail Virtue.

This requires synthesizing rules from both the Bretonnia Army Book and the general Warhammer rulebook.




Subquestions:
- Does losing a combat trigger a break test for any unit, regardless of type?
- Are Grail Knights immune to psychological effects such as panic or fear, and if so, under what conditions?
- What is the effect of a Green Dragon's presence on units during combat, especially regarding fear or panic?


In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from meeplemate.qa_graph import (
    build_question_answer_graph,
    build_coordinating_agent_graph,
    build_analyze_question_graph
)

analyze_graph = build_analyze_question_graph(
    InMemorySaver(),
    chat_model,
    tokenizer,
)

qa_graph = build_question_answer_graph(
    checkpoint_saver=InMemorySaver(),
    chat_model=chat_model,
    tokenizer=tokenizer,
)

coord_graph = build_coordinating_agent_graph(
    checkpoint_saver=InMemorySaver(),
    analyze_question_agent=analyze_graph,
    game_agent=qa_graph,
)

In [ ]:
from typing import cast
from meeplemate.ingest.gamepackage import Manifest
from meeplemate.qa_graph import GameAgentContext, CoordinationInputState
from langchain_core.runnables import RunnableConfig
from uuid import uuid4

input: CoordinationInputState = {
    # "query": "When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?",
    "query": "What is a break test?",
}

context: GameAgentContext = GameAgentContext(
    manifest=cast(Manifest, manifest),
    chunk_search_service=chunk_search_service,
    full_page_store=full_page_store
)
config: RunnableConfig = {"configurable": {"thread_id": str(uuid4())}}

result = await coord_graph.ainvoke(input, context=context, config=config)

display(Markdown(result["response"]))

for item in result["evidence"]:
    display(Markdown(item["content"]))

2026-02-10 20:01:30 [info     ] Analyzing question             query='When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?'


2026-02-10 20:01:31 [info     ] search_chunks called           search_terms=['Grail Knights rules for combat resolution', 'Green Dragon combat mechanics', 'Break test rules for Grail Knights after combat', 'effects of losing combat for Grail Knights', 'conditions under which Grail Knights must take a break test'] token_budget=17291
2026-02-10 20:01:32 [info     ] Used tokens                    query=['Grail Knights rules for combat resolution', 'Green Dragon combat mechanics', 'Break test rules for Grail Knights after combat', 'effects of losing combat for Grail Knights', 'conditions under which Grail Knights must take a break test'] tokens_used=11139 total_retrieved_count=23
2026-02-10 20:01:32 [info     ] Retrieved results              query='When a unit of Grail Knights loses combat against a Green Dragon do the Grail Knights need to take a break test?' relevant_count=23 total_retrieved_count=23
2026-02-10 20:01:37 [info     ] Subquestion identified         explanation='This subques

**No, Grail Knights must take a Break test when they lose combat, even against a Green Dragon.**

The general rule for losing combat requires a Break test:

> The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away. This is called a Break test. You need to take a separate Break test for every unit involved in the combat. (Warhammer Rulebook, p. 42)

This rule applies universally unless overridden by a specific exception.

However, the Grail Knights have a special ability that grants immunity from psychological effects:

> Grail Knights have the most noble chivalric virtue of all – the Grail Virtue. This means that they are unaffected by any of the psychology rules; any such tests they are called upon to take are disregarded with a cool and steely countenance. The Knight knows neither fear nor terror, nor will he panic, for the grail sustains his noble will better than any magic trickery. (Bretonnia Army Book, p. 44)

This exemption, however, only applies to “psychology rules” and “any such tests they are called upon to take” within that category.

A critical distinction is made in the Warhammer Rulebook:

> However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests. (Warhammer Rulebook, p. 47)

Since Break tests are explicitly stated to be *not* psychology tests, the Grail Virtue’s immunity does not extend to Break tests.

Therefore, despite their immunity to fear, terror, and panic, Grail Knights are not exempt from taking a Break test when they lose combat. The general rule takes precedence over the exception because the exception does not apply to the specific mechanic in question.

**Conclusion**: Grail Knights must take a Break test after losing combat, regardless of the enemy type, including against a Green Dragon.

Players will immediately realise that a psychology test is taken in the same way as a Break test in hand- to- hand combat and uses the same characteristic, namely Leadership. However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.

## Grail Virtue

Grail Knights have the most noble chivalric virtue of all - the Grail Virtue. This means that they are unaffected by any of the psychology rules; any such tests they are called upon to take are disregarded with a cool and steely countenance. The Knight knows neither fear nor terror, nor will he panic, for the grail sustains his noble will better than any magic trickery.

## LOSERS TAKE A BREAK TEST

The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away This is called a Break test. You need to take a separate Break test for every unit involved in the combat. Depending on which units pass and which fail their test, some may break and flee whilst others stand their ground. Troops which are better led, braver, and more professional are more likely to stand firm, whilst wild, temperamental troops are far more likely to run for it.

In [7]:
print(result["response"])

**Yes, Grail Knights must take a Break test when they lose combat, despite their immunity to psychological effects.**

The general rule for losing combat requires a Break test:

> The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away. This is called a Break test. You need to take a separate Break test for every unit involved in the combat. (Warhammer Rulebook, p. 42)his means that any unit which loses a combat must attempt a Break test, regardless of other traits.

However, Grail Knights possess the Grail Virtue, which grants immunity to psychological effects:

> Grail Knights have the most noble chivalric virtue of all – the Grail Virtue. This means that they are unaffected by any of the psychology rules; any such tests they are called upon to take are disregarded with a cool and steely countenance. The Knight knows neither fear nor terror, nor will he panic, for the grail sustains his noble will better than any magic trick

In [7]:
from meeplemate.qa_graph import dump_chunks

for question in result["clarifying_questions"]:
    print("Clarifying question:", question["question"])
    # Display answer in markdown
    display(Markdown(question["answer"]))
    for item in question["evidence"]:
        display(Markdown(item["content"]))
    print("\n\n")

Clarifying question: How does Break test interacts with losing combat?


**The Break test is the direct mechanical consequence of losing a combat and must be taken by every unit that loses a hand-to-hand combat.**

> The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away. This is called a Break test. You need to take a separate Break test for every unit involved in the combat. (Warhammer Rulebook, p. 42)

This rule establishes that the Break test is not an optional or secondary mechanic—it is the immediate and mandatory response triggered solely by the act of losing a combat. No other condition or exception is listed that would prevent this test from being taken.

The outcome of the Break test determines whether the unit breaks and flees:
> If the result is greater than the unit's Leadership value then the unit is broken. (Warhammer Rulebook, p. 47)

This confirms that the test is evaluated strictly against the unit’s Leadership value and the dice roll, with no additional modifiers beyond the combat score difference (which is applied as a modifier to the dice roll).

Furthermore, the rules clarify that Break tests are taken *before* Panic tests:
> Panic tests must be taken once all Break tests are complete but before fleeing troops are moved. (Warhammer Rulebook, p. 42)

This sequencing confirms that the Break test is not just a part of the process—it is the first and defining step after losing combat.

Therefore, the interaction between losing combat and the Break test is not one of mechanics overlapping; it is one of direct causation. Losing a combat *triggers* the Break test, which then determines whether the unit flees or remains in place.

In summary: **Losing a combat forces a unit to take a Break test. The Break test is not a separate interaction—it is the formalized resolution of the loss.**

## PANIC TESTS FOR BREAKS

Once all defeated units have taken a Break test, then each remaining unit within \(12^{\circ}\) of friendly units which have broken or been wiped out is called upon to take a Panic test, as described in the Psychology section. This represents the spread of panic amongst the army as friendly units collapse and turn tail. Panic is a special psychological effect, and the full rules for panic are covered in the following section of the rules. However, it is worth bearing in mind at this stage that Panic tests must be taken once all Break tests are complete but before fleeing troops are moved.

## TAKING PSYCHOLOGY TESTS

When taking psychology tests roll 2D6 and compare the result to your Leadership (Ld) value. If the result is less than or equal to the unit's Leadership score the test is passed and all is well. If the result is greater than the unit's Leadership score then the test is failed.

## LOSERS TAKE A BREAK TEST

The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away This is called a Break test. You need to take a separate Break test for every unit involved in the combat. Depending on which units pass and which fail their test, some may break and flee whilst others stand their ground. Troops which are better led, braver, and more professional are more likely to stand firm, whilst wild, temperamental troops are far more likely to run for it.

...

## SPECIAL RULES

## SKELETONS IN COMBAT

Skeletons cannot be broken in hand- to- hand combat and never take Break tests if defeated. Instead their defeat weakens their magic, and for each point by which they lose, one extra Skeleton is removed.

## PURSUIT

If their enemies flee then Ghouls will not pursue, but stop to feed upon the corpses or explore their remains. They will do nothing until they stop feeding. They will automatically stop feeding if charged by an enemy. They will also stop feeding on the D6 roll of a 4 or more if there are enemy within \(12^{\prime \prime}\) at the start of their turn. If no enemy are within \(12^{\prime \prime}\) the Ghouls will continue to feed indefinitely.

## GHOULS IN COMBAT

Ghouls are so cowardly that they will always flee if beaten in combat. No Break test is required as they are assumed to have failed.

## FEAR

FEARUndead cause fear in their enemies as described in the Warhammer rulebook.

## ZOMBIES IN COMBAT

A unit of Zombies is utterly destroyed if it fails a Break test in hand- to- hand combat. The magical bonds that animate it are broken and the corpses fall lifelessly to the ground.

## IMMUNE TO PSYCHOLOGY

Undead are not affected by psychology of any kind and are therefore immune to fear, terror, panic and all other psychology described in the rulebook.




Clarifying question: How does Combat loss condition interacts with Break test?


A unit that loses a combat must take a Break test to determine whether it breaks and flees.

> The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away This is called a Break test.
> 
> (Warhammer Rulebook, p. 42)

This rule establishes that losing a combat directly triggers the Break test. There are no exceptions or alternative outcomes described in the documents that would prevent this from occurring. The Break test is a mandatory consequence of combat loss, regardless of the unit's Leadership value or any other factors.

> Once all defeated units have taken a Break test, then each remaining unit within 12" of friendly units which have broken or been wiped out is called upon to take a Panic test...
> 
> (Warhammer Rulebook, p. 42)

This confirms the sequence: combat loss → Break test → potential Panic tests. The timing and causality are unambiguous—combat loss initiates the Break test process.

Therefore, the condition of losing a combat is the direct cause for taking a Break test. No other mechanism or rule overrides this relationship. The Break test is not optional and occurs immediately after the combat result is determined.

## PANIC TESTS FOR BREAKS

Once all defeated units have taken a Break test, then each remaining unit within \(12^{\circ}\) of friendly units which have broken or been wiped out is called upon to take a Panic test, as described in the Psychology section. This represents the spread of panic amongst the army as friendly units collapse and turn tail. Panic is a special psychological effect, and the full rules for panic are covered in the following section of the rules. However, it is worth bearing in mind at this stage that Panic tests must be taken once all Break tests are complete but before fleeing troops are moved.

## LOSERS TAKE A BREAK TEST

The side that loses a combat must take a test to determine whether it stands and fights or turns tail and runs away This is called a Break test. You need to take a separate Break test for every unit involved in the combat. Depending on which units pass and which fail their test, some may break and flee whilst others stand their ground. Troops which are better led, braver, and more professional are more likely to stand firm, whilst wild, temperamental troops are far more likely to run for it.

In [7]:
from uuid import uuid4
from langchain_core.globals import set_debug

# set_debug(True)

input = {
    "manifest": manifest,
    "query": "How does Grail Virtue interact Break test?",
    "recursion_depth": 0,
    "evidence": [],
    "messages": [],
}

results = []
for _ in range(10):
    responses = await qa_service.abatch([input]*10, config={"configurable": {"thread_id": str(uuid4())}})

    for response in responses:
        result = response["response"].splitlines()[0]
        results.append(result)
        print(result)
        print()

# # Display response as markdown
# from IPython.display import Markdown, display
# display(Markdown(response["response"]))

2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Starting with data retrieval  
2026-02-04 19:39:24 [info     ] Retrieving data for query      query='How does Grail Virtue interact Break test?'
2026-02-04 19:39:24 [info     ] Retrieving data for query      query='How does Grail Virtue interact Break test?'
2026-02-04 19:39:24 [info     ] Retrieving data for query      query='How does Grail Virtue interact Break test?'
2026-02-04 19:39:24 [info   

CancelledError: 

In [ ]:
# Right 95% of the time.
for result in sorted(results):
    print(result)

**Grail Knights are not exempt from Break tests, despite their Grail Virtue.**
**No, Grail Knights are not exempt from Break tests, even though they are unaffected by psychology rules.**
**No, Grail Knights are not exempt from Break tests, even though they have the Grail Virtue.**
**No, Grail Knights are not exempt from Break tests.**
**No, Grail Knights are not exempt from Break tests.**
**No, Grail Knights are not protected from Break tests by the Grail Virtue.**
**No, Grail Knights are not protected from Break tests by the Grail Virtue.**
**No, Grail Knights do not automatically avoid Break tests, even though they are unaffected by psychology.**
**No, Grail Knights do not bypass Break tests due to Grail Virtue.**
**No, Grail Knights do not bypass Break tests, even though they are granted immunity to psychology.**
**No, Grail Knights do not bypass Break tests, even though they possess the Grail Virtue.**
**No, Grail Knights do not gain immunity to Break tests from the Grail Virtue.**

In [13]:
for chunk in response["evidence"]:
    header = f"### From {chunk['rulebook_name']} page {chunk['page']}\n"
    display(Markdown(header + chunk["content"]))


### From Warhammer Rulebook page 47
Players will immediately realise that a psychology test is taken in the same way as a Break test in hand- to- hand combat and uses the same characteristic, namely Leadership. However, a Break test is not a psychology test. The two tests are quite separate. This is important because some bonuses apply specifically to Break tests and others apply specifically to psychology tests.